$$MSE(w) = \frac{1}{n} \sum_{i=1}^{n} (y_i - x_i^\top w)^2\\
J_{\text{ridge}}(w) = MSE(w) + \lambda \|w\|_2^2 = MSE(w) + \lambda \sum_{j=1}^{p} w_j^2\\
J_{\text{lasso}}(w) = MSE(w) + \lambda \|w\|_1 = MSE(w) + \lambda \sum_{j=1}^{p} |w_j|\\
\text{IDF}(t) = \log\left(\frac{N}{n_t}\right)\\
\text{TF-IDF}(t,d) = \text{TF}(t,d) \times \text{IDF}(t)\\
x_i' =  \frac{x_i -\mu}\sigma\\
x_i' = \frac{x_i - x_{\min}}{x_{\max} - x_{\min}}
$$

In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import(
    train_test_split,KFold,cross_val_score,
    learning_curve,GridSearchCV,RandomizedSearchCV
)
from sklearn.preprocessing import(
    StandardScaler,MinMaxScaler,OneHotEncoder,LabelEncoder
)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import(
    SelectKBest,f_regression,RFE,mutual_info_regression
)
from sklearn.linear_model import(
    LinearRegression,Ridge,Lasso,ElasticNet
)
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error,r2_score,mean_absolute_error
#加载数据
california = fetch_california_housing()
df = pd.DataFrame(california.data,columns=california.feature_names)
df['Price'] = california.target

print(f"原始数据形状: {df.shape}")
print(f"\n前5行:\n{df.head()}")
print(f"\n数据类型:\n{df.dtypes}")
print(f"\n统计描述:\n{df.describe()}")

#人为添加脏数据和多种特征类型
np.random.seed(42)
n = len(df)

house_types = ['公寓', '别墅', '联排', '平房', '复式']
df['HouseType'] = np.random.choice(house_types,n,p=[0.35,0.1,0.2,0.25,0.1])

renvation_levels = ['毛坯', '简装', '精装', '豪华']
df['Renovation'] = np.random.choice(renvation_levels,n,p=[0.2, 0.35, 0.3, 0.15])

orientations = ['南', '北', '东', '西', '南北通']
df['Orientation'] = np.random.choice(orientations,n)

原始数据形状: (20640, 9)

前5行:
   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3  5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4  3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   

   Longitude  Price  
0    -122.23  4.526  
1    -122.22  3.585  
2    -122.24  3.521  
3    -122.25  3.413  
4    -122.25  3.422  

数据类型:
MedInc        float64
HouseAge      float64
AveRooms      float64
AveBedrms     float64
Population    float64
AveOccup      float64
Latitude      float64
Longitude     float64
Price         float64
dtype: object

统计描述:
             MedInc      HouseAge      AveRooms     AveBedrms    Population  \
count  20640.000000  20640.000000  20640.000000  20640.000000  20640.000000   
me

In [6]:
# ============================================================
# 5. 项目总结 & 知识地图
# ============================================================

print("""
╔══════════════════════════════════════════════════════════════╗
║                  📝 项目总结报告                         ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  1️⃣  正则化技术                                              ║
║  ├─ Ridge (L2): 系数缩小但不归零 → 防止过拟合               ║
║  ├─ Lasso (L1): 系数可精确归零 → 自动特征选择               ║
║  ├─ ElasticNet: 结合两者优点                                ║
║  └─ 几何解释: L1菱形约束的「角」导致稀疏                     ║
║                                                              ║
║  2️⃣  特征工程                                               ║
║  ├─ 数值缩放: StandardScaler vs MinMaxScaler                ║
║  ├─ 类别编码: OneHot(无序) vs LabelEncoding(有序)           ║
║  ├─ 文本特征: TF-IDF 提取关键词权重                         ║
║  ├─ 特征选择: 相关性/L1/SelectKBest/RFE                     ║
║  └─ 核心原则: 用训练集 fit，训练集+测试集 transform           ║
║                                                              ║
║  3️⃣  模型选择与验证                                         ║
║  ├─ K折CV > 单一验证集（更稳定、更充分利用数据）            ║
║  ├─ 学习曲线: 诊断过拟合/欠拟合的利器                       ║
║  ├─ GridSearch: 穷举搜索，适合小参数空间                    ║
║  └─ RandomSearch: 随机采样，适合大参数空间                   ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
""")

# 保存结果
print("✅ 结果已保存到 model_comparison_results.csv")
print("✅ 项目完成！")


╔══════════════════════════════════════════════════════════════╗
║                  📝 项目总结报告                         ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  1️⃣  正则化技术                                              ║
║  ├─ Ridge (L2): 系数缩小但不归零 → 防止过拟合               ║
║  ├─ Lasso (L1): 系数可精确归零 → 自动特征选择               ║
║  ├─ ElasticNet: 结合两者优点                                ║
║  └─ 几何解释: L1菱形约束的「角」导致稀疏                     ║
║                                                              ║
║  2️⃣  特征工程                                               ║
║  ├─ 数值缩放: StandardScaler vs MinMaxScaler                ║
║  ├─ 类别编码: OneHot(无序) vs LabelEncoding(有序)           ║
║  ├─ 文本特征: TF-IDF 提取关键词权重                         ║
║  ├─ 特征选择: 相关性/L1/SelectKBest/RFE                     ║
║  └─ 核心原则: 用训练集 fit，训练集+测试集 transform           ║
║                                                              ║
║  3️⃣  模型选择